# Creative Memory Copilot - Join multimodal

Objetivo: construir la tabla tabular definitiva para CBR/modelos/dashboard uniendo:

- `feature/processed/creative_feature_base_text_selected_age.csv` como base tabular final.
- Parquets de `fe_vision_v2/outputs/` para features visuales interpretables y embeddings PCA.
- `cv/output/creative_spatial_features.parquet` y JSONL de `cv` como evidencia visual/espacial adicional.

Decision de arquitectura: no convertir todo a JSON. El dataset de modelado se mantiene en Parquet/CSV; JSONL queda como sidecar de evidencia anidada para LLM y UI.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError("Instala pyarrow para leer/escribir Parquet: .venv/bin/pip install -r fe_vision_v2/requirements.txt") from exc

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "feature" / "processed").exists():
    ROOT = ROOT.parent

FEATURE_DIR = ROOT / "feature" / "processed"
FE_VISION_DIR = ROOT / "fe_vision_v2" / "outputs"
CV_DIR = ROOT / "cv" / "output"
DATASET_DIR = ROOT / "dataset"
OUTPUT_DIR = DATASET_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_PATH = FEATURE_DIR / "creative_feature_base_text_selected_age.csv"
FE_VISUAL_PCA_PATH = FE_VISION_DIR / "creative_visual_features_with_embeddings_pca.parquet"
FE_CLIP_PATH = FE_VISION_DIR / "creative_clip_embeddings.parquet"
FE_CNN_PATH = FE_VISION_DIR / "creative_cnn_embeddings.parquet"
IMAGE_QUALITY_PATH = FE_VISION_DIR / "image_quality_report.csv"
CV_SPATIAL_PATH = CV_DIR / "creative_spatial_features.parquet"
CV_PROMPT_JSONL_PATH = CV_DIR / "creative_prompt_analysis.jsonl"
CV_ELEMENTS_JSONL_PATH = CV_DIR / "creative_spatial_elements.jsonl"

TABULAR_VISUAL_PATH = OUTPUT_DIR / "creative_feature_base_tabular_visual_definitive.parquet"
TABULAR_VISUAL_CSV_PATH = OUTPUT_DIR / "creative_feature_base_tabular_visual_definitive.csv"
TABULAR_PLUS_CV_PATH = OUTPUT_DIR / "creative_feature_base_tabular_visual_cv.parquet"
TABULAR_PLUS_CV_CSV_PATH = OUTPUT_DIR / "creative_feature_base_tabular_visual_cv.csv"
CV_PROMPT_FLAT_PATH = OUTPUT_DIR / "creative_cv_prompt_flat.parquet"
CV_EVIDENCE_JSONL_PATH = OUTPUT_DIR / "creative_cv_evidence.jsonl"

paths = {
    "base": BASE_PATH,
    "fe_visual_pca": FE_VISUAL_PCA_PATH,
    "fe_clip_raw": FE_CLIP_PATH,
    "fe_cnn_raw": FE_CNN_PATH,
    "image_quality": IMAGE_QUALITY_PATH,
    "cv_spatial": CV_SPATIAL_PATH,
    "cv_prompt_jsonl": CV_PROMPT_JSONL_PATH,
    "cv_elements_jsonl": CV_ELEMENTS_JSONL_PATH,
}

pd.DataFrame([{"name": name, "exists": path.exists(), "path": str(path)} for name, path in paths.items()])


## 1. Carga e inspeccion de fuentes

`creative_feature_base_text_selected_age.csv` es la base tabular. `fe_vision_v2` aporta dos cosas distintas:

- features visuales interpretables: color, layout, complejidad, masa visual, simetria, etc.;
- embeddings reducidos por PCA: `clip_pca_*` y `cnn_pca_*`.

Los embeddings completos `clip_*` y `cnn_*` se validan, pero no se meten por defecto en la tabla final porque son muy anchos. Para similitud visual real conviene usarlos como vector store separado.

In [ ]:
def assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def assert_unique_key(df: pd.DataFrame, name: str, key: str = "creative_id") -> None:
    if key not in df.columns:
        raise KeyError(f"{name} missing key column: {key}")
    nulls = int(df[key].isna().sum())
    dups = int(df.duplicated(key).sum())
    if nulls or dups:
        raise ValueError(f"{name} key issue: nulls={nulls}, duplicated={dups}")


def source_report(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, table in tables.items():
        rows.append(
            {
                "source": name,
                "rows": len(table),
                "columns": table.shape[1],
                "unique_creative_ids": table["creative_id"].nunique(dropna=False) if "creative_id" in table.columns else np.nan,
                "duplicated_creative_ids": int(table.duplicated("creative_id").sum()) if "creative_id" in table.columns else np.nan,
                "missing_cells": int(table.isna().sum().sum()),
            }
        )
    return pd.DataFrame(rows)


for label, path in paths.items():
    assert_exists(path, label)

base = pd.read_csv(BASE_PATH)
base = base.drop(columns=["asset_path"], errors="ignore")
fe_visual_pca = pd.read_parquet(FE_VISUAL_PCA_PATH)
clip_raw = pd.read_parquet(FE_CLIP_PATH)
cnn_raw = pd.read_parquet(FE_CNN_PATH)
image_quality = pd.read_csv(IMAGE_QUALITY_PATH)
cv_spatial = pd.read_parquet(CV_SPATIAL_PATH)

for name, table in {
    "base": base,
    "fe_visual_pca": fe_visual_pca,
    "clip_raw": clip_raw,
    "cnn_raw": cnn_raw,
    "image_quality": image_quality,
    "cv_spatial": cv_spatial,
}.items():
    assert_unique_key(table, name)

source_report(
    {
        "base": base,
        "fe_visual_pca": fe_visual_pca,
        "clip_raw": clip_raw,
        "cnn_raw": cnn_raw,
        "image_quality": image_quality,
        "cv_spatial": cv_spatial,
    }
)


## 2. Seleccion de columnas de `fe_vision_v2`

No usamos el parquet final entero tal cual porque contiene muchas columnas duplicadas de metadata/KPIs. La base canonica es `creative_feature_base_text_selected_age.csv`; de `fe_vision_v2` solo traemos:

- features visuales interpretables, con prefijo `vis_`;
- `clip_pca_*` y `cnn_pca_*`;
- estado de calidad de imagen, con prefijo `image_quality_`.


In [ ]:
HANDCRAFTED_VISUAL_COLS = [
    "image_area",
    "brightness_mean", "brightness_std", "contrast",
    "saturation_mean", "saturation_std", "hue_mean", "hue_std",
    "dominant_color_r", "dominant_color_g", "dominant_color_b",
    "dominant_color_h", "dominant_color_s", "dominant_color_v",
    "colorfulness", "edge_density", "visual_complexity", "sharpness_laplacian_var",
    "centroid_x_visual_mass", "centroid_y_visual_mass",
    "visual_mass_top_ratio", "visual_mass_bottom_ratio", "visual_mass_left_ratio", "visual_mass_right_ratio",
    "visual_mass_center_ratio", "visual_mass_border_ratio",
    "quadrant_top_left_density", "quadrant_top_right_density", "quadrant_bottom_left_density", "quadrant_bottom_right_density",
    "symmetry_horizontal_score", "symmetry_vertical_score",
    "main_object_bbox_x_min", "main_object_bbox_y_min", "main_object_bbox_x_max", "main_object_bbox_y_max",
    "main_object_bbox_area_ratio", "main_object_center_x", "main_object_center_y", "main_object_is_centered",
    "saliency_center_bias", "background_uniformity_score", "empty_space_ratio",
]

visual_cols = [c for c in HANDCRAFTED_VISUAL_COLS if c in fe_visual_pca.columns]
clip_pca_cols = sorted([c for c in fe_visual_pca.columns if c.startswith("clip_pca_")], key=lambda c: int(c.rsplit("_", 1)[1]))
cnn_pca_cols = sorted([c for c in fe_visual_pca.columns if c.startswith("cnn_pca_")], key=lambda c: int(c.rsplit("_", 1)[1]))

fe_keep = fe_visual_pca[["creative_id", *visual_cols, *clip_pca_cols, *cnn_pca_cols]].copy()
fe_keep = fe_keep.rename(columns={c: f"vis_{c}" for c in visual_cols})

quality_keep = image_quality[["creative_id", "status", "reason", "width", "height"]].copy()
quality_keep = quality_keep.rename(
    columns={
        "status": "image_quality_status",
        "reason": "image_quality_reason",
        "width": "image_quality_width",
        "height": "image_quality_height",
    }
)

pd.DataFrame(
    [
        {"family": "handcrafted_visual", "columns": len(visual_cols)},
        {"family": "clip_pca", "columns": len(clip_pca_cols)},
        {"family": "cnn_pca", "columns": len(cnn_pca_cols)},
        {"family": "raw_clip_embedding_sidecar", "columns": clip_raw.shape[1] - 1},
        {"family": "raw_cnn_embedding_sidecar", "columns": cnn_raw.shape[1] - 1},
    ]
)


In [ ]:
tabular_visual = (
    base.merge(fe_keep, on="creative_id", how="left", validate="1:1")
    .merge(quality_keep, on="creative_id", how="left", validate="1:1")
)

coverage_cols = [*clip_pca_cols, *cnn_pca_cols, *[f"vis_{c}" for c in visual_cols]]
coverage_report = pd.DataFrame(
    [
        {
            "check": "rows_preserved",
            "value": len(tabular_visual) == len(base),
            "detail": f"base={len(base)}, joined={len(tabular_visual)}",
        },
        {
            "check": "unique_creative_id",
            "value": tabular_visual["creative_id"].nunique() == len(tabular_visual),
            "detail": f"unique={tabular_visual['creative_id'].nunique()}, rows={len(tabular_visual)}",
        },
        {
            "check": "rows_with_fe_vision_values",
            "value": int(tabular_visual[coverage_cols].notna().any(axis=1).sum()),
            "detail": f"out_of={len(tabular_visual)}",
        },
        {
            "check": "image_quality_ok",
            "value": int(tabular_visual["image_quality_status"].eq("ok").sum()),
            "detail": f"out_of={len(tabular_visual)}",
        },
    ]
)

tabular_visual.shape, coverage_report


## 3. Como juntar con `cv`

`cv` tiene dos tipos de salida:

1. `creative_spatial_features.parquet`: tabla plana, lista para join por `creative_id`.
2. JSONL: estructuras anidadas con OCR, bounding boxes, elementos detectados, paletas, layouts y evidencia rica.

Conclusion: no hace falta pasarlo todo a JSON. Para modelos y CBR usamos columnas planas; para el LLM y UI mantenemos JSONL como evidencia consultable por `creative_id`.

In [ ]:
CV_META_COLS = {"asset_file", "vertical", "format", "advertiser_name", "asset_path"}
cv_feature_cols = [c for c in cv_spatial.columns if c != "creative_id" and c not in CV_META_COLS]
cv_keep = cv_spatial[["creative_id", *cv_feature_cols]].copy()
cv_keep = cv_keep.rename(columns={c: f"cv_{c}" for c in cv_feature_cols})

tabular_plus_cv_spatial = tabular_visual.merge(cv_keep, on="creative_id", how="left", validate="1:1")

pd.DataFrame(
    [
        {"object": "cv_spatial_flat_features", "columns": len(cv_feature_cols), "rows_with_values": int(tabular_plus_cv_spatial[[f'cv_{c}' for c in cv_feature_cols]].notna().any(axis=1).sum())},
        {"object": "tabular_visual_before_cv", "columns": tabular_visual.shape[1], "rows": len(tabular_visual)},
        {"object": "tabular_plus_cv_spatial", "columns": tabular_plus_cv_spatial.shape[1], "rows": len(tabular_plus_cv_spatial)},
    ]
)


## 4. JSONL de `cv`: evidencia para LLM y dashboard

`creative_prompt_analysis.jsonl` ya trae un resumen estructurado por creative. `creative_spatial_elements.jsonl` trae OCR/grounding con cajas y texto. En vez de expandirlo todo a cientos de columnas, hacemos dos cosas:

- extraemos una version plana pequena de `creative_prompt_analysis.jsonl` para features seleccionadas;
- guardamos un sidecar `creative_cv_evidence.jsonl` que conserva el JSON anidado por creative para explicaciones y UI.

In [ ]:
CREATIVE_ID_RE = re.compile(r"(\d+)$")


def normalize_creative_id(value: Any) -> int | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (int, np.integer)):
        return int(value)
    text = str(value)
    match = CREATIVE_ID_RE.search(text)
    return int(match.group(1)) if match else None


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def nested_get(obj: dict[str, Any], path: list[str], default: Any = np.nan) -> Any:
    cur: Any = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur or cur[key] is None:
            return default
        cur = cur[key]
    return cur


def bool_to_int(value: Any) -> Any:
    if value is np.nan or value is None:
        return np.nan
    return int(bool(value))


prompt_rows = read_jsonl(CV_PROMPT_JSONL_PATH)
elements_rows = read_jsonl(CV_ELEMENTS_JSONL_PATH)

for row in prompt_rows:
    row["creative_id"] = normalize_creative_id(row.get("creative_id"))
for row in elements_rows:
    row["creative_id"] = normalize_creative_id(row.get("creative_id"))

prompt_by_id = {row["creative_id"]: row for row in prompt_rows if row.get("creative_id") is not None}
elements_by_id = {row["creative_id"]: row for row in elements_rows if row.get("creative_id") is not None}

pd.DataFrame(
    [
        {"jsonl": "prompt_analysis", "rows": len(prompt_rows), "unique_ids": len(prompt_by_id)},
        {"jsonl": "spatial_elements", "rows": len(elements_rows), "unique_ids": len(elements_by_id)},
    ]
)


In [ ]:
def flatten_prompt_analysis(obj: dict[str, Any]) -> dict[str, Any]:
    text_elements = obj.get("text_elements") or []
    detected = obj.get("detected_elements") or {}
    visual_elements = obj.get("visual_elements") or {}

    flat = {
        "creative_id": normalize_creative_id(obj.get("creative_id")),
        "prompt_image_width": nested_get(obj, ["image_dimensions", "width"]),
        "prompt_image_height": nested_get(obj, ["image_dimensions", "height"]),
        "prompt_image_aspect_ratio": nested_get(obj, ["image_dimensions", "aspect_ratio"]),
        "prompt_text_element_count": len(text_elements),
        "prompt_text_total_area": sum(float(el.get("area_relative") or 0) for el in text_elements if isinstance(el, dict)),
        "prompt_detected_brand_logo": bool_to_int(bool(detected.get("brand_logo"))),
        "prompt_detected_primary_cta": bool_to_int(bool(detected.get("primary_cta"))),
        "prompt_detected_promo_badge": bool_to_int(bool(detected.get("promo_badge"))),
        "prompt_detected_product_area": bool_to_int(bool(detected.get("product_area"))),
        "prompt_detected_social_proof": bool_to_int(bool(detected.get("social_proof"))),
        "prompt_visual_element_count": visual_elements.get("count", np.nan),
        "prompt_visual_card_count": visual_elements.get("card_count", np.nan),
        "prompt_visual_circle_count": visual_elements.get("circle_count", np.nan),
        "prompt_visual_total_area": visual_elements.get("total_area", np.nan),
        "prompt_visual_grid_like": bool_to_int(visual_elements.get("grid_like")),
        "prompt_visual_center_y": visual_elements.get("center_y", np.nan),
        "prompt_dominant_color_name": nested_get(obj, ["color_features", "dominant_color_name"]),
        "prompt_dominant_color_hex": nested_get(obj, ["color_features", "dominant_color_hex"]),
        "prompt_dominant_color_proportion": nested_get(obj, ["color_features", "dominant_color_proportion"]),
        "prompt_background_color_name": nested_get(obj, ["color_features", "background_color_name"]),
        "prompt_background_color_hex": nested_get(obj, ["color_features", "background_color_hex"]),
        "prompt_background_is_solid": bool_to_int(nested_get(obj, ["color_features", "background_is_solid"], None)),
        "prompt_saturation_mean": nested_get(obj, ["color_features", "saturation_mean"]),
        "prompt_brightness_mean": nested_get(obj, ["color_features", "brightness_mean"]),
        "prompt_color_warmth": nested_get(obj, ["color_features", "color_warmth"]),
        "prompt_contrast_score": nested_get(obj, ["color_features", "contrast_score"]),
        "prompt_layout_template": nested_get(obj, ["layout_classification", "template"]),
        "prompt_has_grid_layout": bool_to_int(nested_get(obj, ["layout_classification", "has_grid_layout"], None)),
        "prompt_has_carousel_indicator": bool_to_int(nested_get(obj, ["layout_classification", "has_carousel_indicator"], None)),
        "prompt_brand_zone_ratio": nested_get(obj, ["content_zones", "brand_zone_ratio"]),
        "prompt_showcase_zone_ratio": nested_get(obj, ["content_zones", "showcase_zone_ratio"]),
        "prompt_copy_zone_ratio": nested_get(obj, ["content_zones", "copy_zone_ratio"]),
        "prompt_cta_zone_ratio": nested_get(obj, ["content_zones", "cta_zone_ratio"]),
        "prompt_total_text_area_ratio": nested_get(obj, ["content_zones", "total_text_area_ratio"]),
        "prompt_headline_area_ratio": nested_get(obj, ["content_zones", "headline_area_ratio"]),
        "prompt_headline_is_large": bool_to_int(nested_get(obj, ["content_zones", "headline_is_large"], None)),
        "prompt_vertical_center_of_mass": nested_get(obj, ["content_zones", "vertical_center_of_mass"]),
        "prompt_background_edge_density": nested_get(obj, ["background", "background_edge_density"]),
        "prompt_has_decorative_pattern": bool_to_int(nested_get(obj, ["background", "has_decorative_pattern"], None)),
        "prompt_cta_y_relative": nested_get(obj, ["spatial_features", "cta_y_relative"]),
        "prompt_cta_x_relative": nested_get(obj, ["spatial_features", "cta_x_relative"]),
        "prompt_cta_in_bottom_third": bool_to_int(nested_get(obj, ["spatial_features", "cta_in_bottom_third"], None)),
        "prompt_badge_in_top_right": bool_to_int(nested_get(obj, ["spatial_features", "badge_in_top_right"], None)),
        "prompt_logo_in_top_left": bool_to_int(nested_get(obj, ["spatial_features", "logo_in_top_left"], None)),
        "prompt_product_zone_coverage": nested_get(obj, ["spatial_features", "product_zone_coverage"]),
        "prompt_text_to_visual_ratio": nested_get(obj, ["spatial_features", "text_to_visual_ratio"]),
        "prompt_layout_pattern_type": nested_get(obj, ["layout_pattern", "type"]),
        "prompt_layout_pattern_confidence": nested_get(obj, ["layout_pattern", "confidence"]),
        "prompt_thumb_zone_cta": nested_get(obj, ["design_conventions", "thumb_zone_cta"]),
        "prompt_visual_hierarchy_score": nested_get(obj, ["design_conventions", "visual_hierarchy_score"]),
        "prompt_badge_visibility_score": nested_get(obj, ["design_conventions", "badge_visibility_score"]),
        "prompt_brand_clarity_score": nested_get(obj, ["design_conventions", "brand_clarity_score"]),
        "prompt_whitespace_balance": nested_get(obj, ["design_conventions", "whitespace_balance"]),
    }
    return flat


prompt_flat = pd.DataFrame([flatten_prompt_analysis(row) for row in prompt_rows])
assert_unique_key(prompt_flat, "prompt_flat")

tabular_plus_cv = tabular_plus_cv_spatial.merge(prompt_flat, on="creative_id", how="left", validate="1:1")

pd.DataFrame(
    [
        {"object": "prompt_flat", "rows": len(prompt_flat), "columns": prompt_flat.shape[1]},
        {"object": "tabular_plus_cv", "rows": len(tabular_plus_cv), "columns": tabular_plus_cv.shape[1]},
        {"object": "rows_with_prompt_features", "rows": int(tabular_plus_cv[prompt_flat.columns.drop('creative_id')].notna().any(axis=1).sum()), "columns": ""},
    ]
)


## 5. Sidecar JSON de evidencia

Este archivo no reemplaza a la tabla. Sirve para cuando el usuario seleccione un creative y el LLM necesite evidencia visual detallada: OCR, cajas, paleta, layout, CTA, zonas, etc.

In [ ]:
with CV_EVIDENCE_JSONL_PATH.open("w", encoding="utf-8") as handle:
    for creative_id in base["creative_id"].astype(int).sort_values():
        evidence = {
            "creative_id": int(creative_id),
            "prompt_analysis": prompt_by_id.get(int(creative_id)),
            "spatial_elements": elements_by_id.get(int(creative_id)),
        }
        handle.write(json.dumps(evidence, ensure_ascii=True))
        handle.write("\n")

evidence_report = pd.DataFrame(
    [
        {"check": "evidence_jsonl_rows", "value": sum(1 for _ in CV_EVIDENCE_JSONL_PATH.open("r", encoding="utf-8")), "path": str(CV_EVIDENCE_JSONL_PATH)},
        {"check": "prompt_ids_available", "value": len(prompt_by_id), "path": str(CV_PROMPT_JSONL_PATH)},
        {"check": "spatial_element_ids_available", "value": len(elements_by_id), "path": str(CV_ELEMENTS_JSONL_PATH)},
    ]
)
evidence_report


## 6. Manifest de columnas y guardado

Guardamos:

- `creative_feature_base_tabular_visual_definitive`: base tabular + `fe_vision_v2` + PCA visual.
- `creative_feature_base_tabular_visual_cv`: lo anterior + features planas de `cv` + prompt flat.
- `creative_cv_prompt_flat`: version plana pequena del JSON de prompt.
- `creative_cv_evidence.jsonl`: evidencia anidada para LLM/UI.


In [ ]:
def infer_column_layer(column: str) -> str:
    if column in base.columns:
        return "base_tabular_text_age"
    if column.startswith("vis_"):
        return "fe_vision_handcrafted"
    if column.startswith("clip_pca_"):
        return "fe_vision_clip_pca"
    if column.startswith("cnn_pca_"):
        return "fe_vision_cnn_pca"
    if column.startswith("image_quality_"):
        return "fe_vision_image_quality"
    if column.startswith("cv_"):
        return "cv_spatial_flat"
    if column.startswith("prompt_"):
        return "cv_prompt_flat"
    return "unknown"


column_manifest = pd.DataFrame(
    [
        {
            "column": column,
            "layer": infer_column_layer(column),
            "dtype": str(tabular_plus_cv[column].dtype),
            "missing_count": int(tabular_plus_cv[column].isna().sum()),
            "unique_count": int(tabular_plus_cv[column].nunique(dropna=False)),
        }
        for column in tabular_plus_cv.columns
    ]
)

join_report = pd.concat(
    [
        coverage_report,
        evidence_report.rename(columns={"path": "detail"}).assign(value=lambda d: d["value"].astype(object)),
        pd.DataFrame(
            [
                {"check": "tabular_visual_shape", "value": True, "detail": f"{tabular_visual.shape}"},
                {"check": "tabular_plus_cv_shape", "value": True, "detail": f"{tabular_plus_cv.shape}"},
                {"check": "raw_clip_embedding_strategy", "value": True, "detail": f"kept as sidecar: {FE_CLIP_PATH.name} ({clip_raw.shape[1] - 1} dims)"},
                {"check": "raw_cnn_embedding_strategy", "value": True, "detail": f"kept as sidecar: {FE_CNN_PATH.name} ({cnn_raw.shape[1] - 1} dims)"},
            ]
        ),
    ],
    ignore_index=True,
)

tabular_visual.to_parquet(TABULAR_VISUAL_PATH, index=False)
tabular_visual.to_csv(TABULAR_VISUAL_CSV_PATH, index=False)
tabular_plus_cv.to_parquet(TABULAR_PLUS_CV_PATH, index=False)
tabular_plus_cv.to_csv(TABULAR_PLUS_CV_CSV_PATH, index=False)
prompt_flat.to_parquet(CV_PROMPT_FLAT_PATH, index=False)
column_manifest.to_csv(OUTPUT_DIR / "multimodal_column_manifest.csv", index=False)
join_report.to_csv(OUTPUT_DIR / "multimodal_join_report.csv", index=False)

pd.DataFrame(
    [
        {"artifact": "tabular_visual_definitive_parquet", "rows": len(tabular_visual), "columns": tabular_visual.shape[1], "path": str(TABULAR_VISUAL_PATH)},
        {"artifact": "tabular_visual_definitive_csv", "rows": len(tabular_visual), "columns": tabular_visual.shape[1], "path": str(TABULAR_VISUAL_CSV_PATH)},
        {"artifact": "tabular_plus_cv_parquet", "rows": len(tabular_plus_cv), "columns": tabular_plus_cv.shape[1], "path": str(TABULAR_PLUS_CV_PATH)},
        {"artifact": "tabular_plus_cv_csv", "rows": len(tabular_plus_cv), "columns": tabular_plus_cv.shape[1], "path": str(TABULAR_PLUS_CV_CSV_PATH)},
        {"artifact": "prompt_flat_parquet", "rows": len(prompt_flat), "columns": prompt_flat.shape[1], "path": str(CV_PROMPT_FLAT_PATH)},
        {"artifact": "cv_evidence_jsonl", "rows": len(base), "columns": None, "path": str(CV_EVIDENCE_JSONL_PATH)},
        {"artifact": "column_manifest", "rows": len(column_manifest), "columns": column_manifest.shape[1], "path": str(OUTPUT_DIR / "multimodal_column_manifest.csv")},
        {"artifact": "join_report", "rows": len(join_report), "columns": join_report.shape[1], "path": str(OUTPUT_DIR / "multimodal_join_report.csv")},
    ]
)


## 7. Recomendacion de uso

- Para rankings, dashboards y modelos tabulares: usar `creative_feature_base_tabular_visual_definitive.parquet`.
- Para explicabilidad espacial adicional: usar `creative_feature_base_tabular_visual_cv.parquet`.
- Para similitud visual real: usar `creative_clip_embeddings.parquet` como vector store; no hace falta pegar 512 columnas al CSV principal.
- Para LLM: recuperar la fila tabular y el objeto de `creative_cv_evidence.jsonl` por `creative_id`.

No pasar todo a JSON: perderiamos eficiencia para modelos, joins, filtros y UMAP. Mejor formato hibrido: Parquet/CSV para features planas, JSONL para evidencia anidada.